# Particle Filtering - Single Pendulum

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from multi_layer_perceptron import (
    simulate,
    particle_filter,
    particle_mean,
    particle_std,
)

In [ ]:
T = 5.0
dt_sim = 1e-3  # 1 kHz
dt_obs = 5e-3  # 200 Hz

D = 2
O = 1

q = 0.1
r = 0.3

# Q = q**2 * jnp.eye(D)
Q = jnp.diag(jnp.array([1e-6, q**2]))
R = r**2 * jnp.eye(O)

g = 9.81
l = 1.0
m = 1.0
b = 0.7

In [ ]:
def f(x, t, dt):
    theta = x[..., 0]
    omega = x[..., 1]
    omega_next = omega + dt * (-(g / l) * jnp.sin(theta) - (b / (m * l**2)) * omega)
    theta_next = theta + dt * omega_next
    return jnp.stack([theta_next, omega_next], axis=-1)


def h(x):
    return x[..., 0:1]

In [ ]:
key = jax.random.PRNGKey(seed=42)

x0 = jnp.array([0.0, 0.0])

(t_sim, x_sim), (t_obs, y_obs) = simulate(f, h, x0, T, dt_sim, dt_obs, Q, R, key)

In [ ]:
N = 5000

particles, log_weights, ess = particle_filter(
    f,
    h,
    t_obs,
    y_obs,
    n_particles=N,
    x0_mean=jnp.zeros(D),
    x0_cov=jnp.eye(D),
    Q=Q,
    R=R,
    key=key,
    n_substeps=5,
)

In [ ]:
mean = particle_mean(particles, log_weights)
std = particle_std(particles, log_weights)

fig, axs = plt.subplots(D+1, 1, figsize=(8, 2.5*(D+1)), sharex=True)
for i in range(D):
    axs[i].plot(t_sim, x_sim[...,i], lw=0.8, alpha=0.7)
    axs[i].plot(t_obs, mean[..., i], lw=1.5, alpha=0.7)
    axs[i].fill_between(t_obs, mean[..., i] - 2 * std[..., i], mean[..., i] + 2 * std[..., i], alpha=0.3, step="mid")
    axs[i].set_ylabel(f"x{i+1}")
    axs[i].grid()
for j in range(O):
    axs[j].scatter(t_obs, y_obs[...,j], marker="x", s=1.0, alpha=0.1, c="r")
axs[-1].plot(t_obs, ess)
axs[-1].axhline(y=0.5 * N, ls="--", c="black", alpha=0.3)
axs[-1].set_ylim(0, 1.1 * N)
axs[-1].set_ylabel("ESS")
axs[-1].set_xlabel("t (s)")
axs[-1].grid(True)
plt.tight_layout();